# Chronos-2 P0a: calibration-only covariate harm screen

Runs `target_only` and `all_dynamic` on **all calibration origins** of the four frozen FEV tasks. It never instantiates sealed evaluation origins and does not search covariate subsets. Results are `screening_only`.

Use an **A100 or H100** if available. A T4 is valid but the Rossmann units will be slower. Every completed task-variant unit is written directly to Google Drive, so rerunning after a disconnect resumes safely.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
FEV_COMMIT = '38007871dcf6dc6b04aed3a54d9cd86678d48d0b'
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'chronos-forecasting==2.2.2',
        f'git+https://github.com/autogluon/fev.git@{FEV_COMMIT}',
    ],
    check=True,
)

SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
covsafe = importlib.import_module('covsafe')
print('Repository ready:', REPO)
print('Git commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip())
print('covsafe import:', covsafe.__file__)

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'Select Runtime > Change runtime type > GPU, then restart.'
)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Torch:', torch.__version__)

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
OUTPUT_ROOT = Path(
    '/content/drive/MyDrive/covariate-safe-tsfm/private_manifests/p0a'
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Durable output root:', OUTPUT_ROOT)

In [ ]:
import json

from covsafe.chronos2_p0a import run_chronos2_p0a
from covsafe.p0a import EXPECTED_P0A_CONFIG_HASH

print('Frozen P0a config hash:', EXPECTED_P0A_CONFIG_HASH)
report = run_chronos2_p0a(REPO, OUTPUT_ROOT)
print(json.dumps(report, indent=2, ensure_ascii=False, default=str))

## Return artifact

Send the final JSON report printed above. If the runtime disconnects, reconnect, select a GPU, and run all cells again; completed units will print `RESUME` and will not be recomputed.